In [ ]:
# ============================================================
# NOTEBOOK 08C — GCN GRAPH ABLATION
# G_native vs G_operational vs G_shuffled
# FROZEN 07D PROTOCOL / NO TEST-BASED SELECTION
#
# Inputs:
#   X_graph_{split}_scaled.npy : [snapshots, 5, 7, 3]
#   y_graph_{split}.npy        : [snapshots, 7]
#   metadata_graph_{split}.csv
#   edge_index_native.npy
#   edge_index_operational.npy
#   edge_index_shuffled_degree_preserving.npy
#
# Target: temperature t+15 min for every node.
# Architecture selection: validation RMSE only on G_operational.
# Same selected architecture is then frozen and used for ALL graph variants.
# Final reporting: 3 predefined seeds (42, 123, 2026).
#
# IMPORTANT:
# - G_native has no inferred inter-device edges. Self-loops are added internally.
# - G_operational uses the train-only operational similarity topology frozen in 07D.
# - G_shuffled preserves the labeled node degree sequence but destroys the
#   operational edge arrangement.
# - No UUID -> IfcSpace spatial claim is introduced.
# - No torch-geometric dependency: fixed 7-node GCN implemented directly in PyTorch.
# ============================================================

from pathlib import Path
import json, random, time, platform
import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader
except ImportError as e:
    raise ImportError("PyTorch is required for 08C.") from e

ROOT = Path.cwd().resolve()
IN = ROOT / "outputs" / "07D_final_alignment"
OUT = ROOT / "outputs" / "08C_gcn_graph_ablation"
OUT.mkdir(parents=True, exist_ok=True)

EXPECTED_SNAPSHOTS = {"train": 9243, "validation": 1975, "test": 1970}
EXPECTED_NODE_LEVEL = {"train": 64701, "validation": 13825, "test": 13790}
NODES = 7
SEQ = 5
CHANNELS = 3
SEEDS = [42, 123, 2026]
BATCH_SIZE = 256
MAX_EPOCHS = 80
PATIENCE = 10
LR = 1e-3
WEIGHT_DECAY = 1e-5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("="*78)
print("NOTEBOOK 08C — GCN GRAPH ABLATION")
print("="*78)
print("Input :", IN)
print("Output:", OUT)
print("Device:", device)
print("PyTorch:", torch.__version__)

# ------------------------------------------------------------
# 1. LOAD FROZEN GRAPH DATA
# ------------------------------------------------------------
data = {}
for split in ["train", "validation", "test"]:
    xp = IN / f"X_graph_{split}_scaled.npy"
    yp = IN / f"y_graph_{split}.npy"
    mp = IN / f"metadata_graph_{split}.csv"
    for p in [xp, yp, mp]:
        assert p.exists(), f"Missing: {p}"

    X = np.load(xp).astype(np.float32)
    y = np.load(yp).astype(np.float32)
    meta = pd.read_csv(mp)

    assert X.shape == (EXPECTED_SNAPSHOTS[split], SEQ, NODES, CHANNELS), (split, X.shape)
    assert y.shape == (EXPECTED_SNAPSHOTS[split], NODES), (split, y.shape)
    assert len(meta) == EXPECTED_SNAPSHOTS[split], (split, len(meta))
    assert y.size == EXPECTED_NODE_LEVEL[split]

    data[split] = (X, y, meta)
    print(f"{split:10s}: X={X.shape}, y={y.shape}, node predictions={y.size}")

# ------------------------------------------------------------
# 2. LOAD FROZEN TOPOLOGIES
# ------------------------------------------------------------
edge_files = {
    "G_native": IN / "edge_index_native.npy",
    "G_operational": IN / "edge_index_operational.npy",
    "G_shuffled": IN / "edge_index_shuffled_degree_preserving.npy",
}
for p in edge_files.values():
    assert p.exists(), f"Missing: {p}"

def canonical_edge_index(arr):
    arr = np.asarray(arr)
    if arr.size == 0:
        return np.empty((2,0), dtype=np.int64)
    if arr.ndim != 2:
        raise ValueError(f"edge_index must be 2D, got {arr.shape}")
    if arr.shape[0] == 2:
        e = arr.astype(np.int64)
    elif arr.shape[1] == 2:
        e = arr.T.astype(np.int64)
    else:
        raise ValueError(f"Unexpected edge_index shape: {arr.shape}")
    assert e.min() >= 0 and e.max() < NODES
    return e

edges = {name: canonical_edge_index(np.load(path)) for name, path in edge_files.items()}

def undirected_pairs(e):
    pairs=set()
    for a,b in e.T:
        a,b=int(a),int(b)
        if a != b:
            pairs.add(tuple(sorted((a,b))))
    return sorted(pairs)

for name,e in edges.items():
    print(f"{name:13s}: stored directed entries={e.shape[1]}, "
          f"unique undirected inter-device edges={len(undirected_pairs(e))}")

assert len(undirected_pairs(edges["G_native"])) == 0, \
    "G_native unexpectedly contains inter-device edges."

# ------------------------------------------------------------
# 3. NORMALIZED ADJACENCY
# ------------------------------------------------------------
# Convert every topology to an undirected binary adjacency, then add self-loops.
# A_hat = D^-1/2 (A + I) D^-1/2
def normalized_adjacency(edge_index):
    A = np.zeros((NODES, NODES), dtype=np.float32)
    for a,b in edge_index.T:
        a,b=int(a),int(b)
        if a != b:
            A[a,b] = 1.0
            A[b,a] = 1.0
    A += np.eye(NODES, dtype=np.float32)
    deg = A.sum(axis=1)
    inv = np.power(deg, -0.5)
    Ahat = inv[:,None] * A * inv[None,:]
    return torch.tensor(Ahat, dtype=torch.float32, device=device)

Ahats = {name: normalized_adjacency(e) for name,e in edges.items()}

# Audit degree preservation independently.
def degree_vector(edge_index):
    A=np.zeros((NODES,NODES),dtype=int)
    for a,b in edge_index.T:
        a,b=int(a),int(b)
        if a!=b: A[a,b]=A[b,a]=1
    return A.sum(axis=1)

deg_op = degree_vector(edges["G_operational"])
deg_sh = degree_vector(edges["G_shuffled"])
assert np.array_equal(deg_op, deg_sh), (deg_op, deg_sh)
print("Operational degrees:", deg_op.tolist())
print("Shuffled degrees   :", deg_sh.tolist())
print("Degree preservation:", bool(np.array_equal(deg_op,deg_sh)))

# ------------------------------------------------------------
# 4. MODEL
# ------------------------------------------------------------
def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass

class GraphConv(nn.Module):
    def __init__(self, fin, fout):
        super().__init__()
        self.lin = nn.Linear(fin, fout)
    def forward(self, x, Ahat):
        # x: [B,N,F]
        z = torch.einsum("ij,bjf->bif", Ahat, x)
        return self.lin(z)

class TemporalGCN(nn.Module):
    """
    Controlled graph model:
    flatten each node's exact 5x3 history -> 15 features,
    then two GCN layers over the 7-node topology.
    This isolates the effect of graph structure while preserving
    the same primary sensor information used by 07D.
    """
    def __init__(self, hidden=32, dropout=0.1):
        super().__init__()
        self.gc1 = GraphConv(SEQ*CHANNELS, hidden)
        self.gc2 = GraphConv(hidden, hidden)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, Ahat):
        # [B,5,7,3] -> [B,7,15]
        x = x.permute(0,2,1,3).contiguous().view(x.size(0), NODES, SEQ*CHANNELS)
        z = torch.relu(self.gc1(x, Ahat))
        z = self.drop(z)
        z = torch.relu(self.gc2(z, Ahat))
        z = self.drop(z)
        return self.head(z).squeeze(-1)  # [B,7]

def loader(split, shuffle=False):
    X,y,_ = data[split]
    ds = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)

def predict(model, split, Ahat):
    model.eval()
    out=[]
    with torch.no_grad():
        for xb,_ in loader(split, False):
            out.append(model(xb.to(device), Ahat).cpu().numpy())
    return np.concatenate(out, axis=0)

def calc_metrics(y,p):
    y=y.reshape(-1); p=p.reshape(-1)
    e=p-y
    mae=float(np.mean(np.abs(e)))
    rmse=float(np.sqrt(np.mean(e**2)))
    sst=float(np.sum((y-y.mean())**2))
    r2=float(1-np.sum(e**2)/sst)
    return {
        "n":len(y), "MAE":mae, "RMSE":rmse, "R2":r2,
        "Bias":float(np.mean(e)),
        "Error_SD":float(np.std(e,ddof=1))
    }

def fit(model, Ahat, seed, save_path):
    seed_all(seed)
    model=model.to(device)
    opt=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    loss_fn=nn.MSELoss()
    best=np.inf; bad=0; history=[]

    for epoch in range(1,MAX_EPOCHS+1):
        model.train()
        losses=[]
        for xb,yb in loader("train", True):
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad()
            pred=model(xb,Ahat)
            loss=loss_fn(pred,yb)
            loss.backward()
            opt.step()
            losses.append(float(loss.item()))

        vp=predict(model,"validation",Ahat)
        vy=data["validation"][1]
        vrmse=float(np.sqrt(np.mean((vp-vy)**2)))
        history.append({
            "epoch":epoch,
            "train_MSE":float(np.mean(losses)),
            "validation_RMSE":vrmse
        })

        if vrmse < best-1e-7:
            best=vrmse; bad=0
            torch.save(model.state_dict(),save_path)
        else:
            bad+=1
            if bad>=PATIENCE:
                break

    model.load_state_dict(
        torch.load(save_path,map_location=device,weights_only=True)
    )
    return model,pd.DataFrame(history),best

# ------------------------------------------------------------
# 5. ARCHITECTURE SELECTION — OPERATIONAL GRAPH, VALIDATION ONLY
# ------------------------------------------------------------
# Topology is NOT selected: all three topologies are required ablations.
# Only hidden width is selected using G_operational validation performance.
candidates = [
    ("GCN_h16", 16),
    ("GCN_h32", 32),
    ("GCN_h64", 64),
]

print("\n[1/3] VALIDATION-ONLY GCN ARCHITECTURE SELECTION")
selection=[]
for name,hidden in candidates:
    seed_all(42)
    model=TemporalGCN(hidden=hidden,dropout=0.1)
    ckpt=OUT/f"_selection_{name}_G_operational.pt"
    model,hist,best=fit(model,Ahats["G_operational"],42,ckpt)
    hist.to_csv(OUT/f"history_selection_{name}_G_operational.csv",index=False)
    row={
        "configuration":name,
        "hidden":hidden,
        "selection_topology":"G_operational",
        "selection_seed":42,
        "best_validation_RMSE":best,
        "epochs_run":len(hist)
    }
    selection.append(row)
    print(row)

sel_df=pd.DataFrame(selection)
sel_df.to_csv(OUT/"01_validation_architecture_selection.csv",index=False)
bestrow=sel_df.sort_values("best_validation_RMSE").iloc[0]
SELECTED_HIDDEN=int(bestrow["hidden"])
SELECTED_NAME=str(bestrow["configuration"])
print("Selected:", SELECTED_NAME)

# ------------------------------------------------------------
# 6. FINAL ABLATION — SAME ARCHITECTURE, ALL TOPOLOGIES, 3 SEEDS
# ------------------------------------------------------------
print("\n[2/3] FINAL GRAPH ABLATION — 3 SEEDS")
all_metrics=[]
all_preds=[]

for topology in ["G_native","G_operational","G_shuffled"]:
    Ahat=Ahats[topology]
    for seed in SEEDS:
        seed_all(seed)
        model=TemporalGCN(hidden=SELECTED_HIDDEN,dropout=0.1)
        ckpt=OUT/f"{topology}_{SELECTED_NAME}_seed{seed}.pt"
        t0=time.time()
        model,hist,best=fit(model,Ahat,seed,ckpt)
        elapsed=time.time()-t0
        hist.to_csv(OUT/f"history_{topology}_seed{seed}.csv",index=False)

        for split in ["train","validation","test"]:
            y=data[split][1]
            p=predict(model,split,Ahat)
            m=calc_metrics(y,p)
            all_metrics.append({
                "model":"GCN",
                "topology":topology,
                "configuration":SELECTED_NAME,
                "seed":seed,
                "split":split,
                **m,
                "best_validation_RMSE_during_training":best,
                "training_seconds":elapsed
            })

            # metadata_graph is snapshot-level. Expand deterministically to node-level.
            meta=data[split][2].copy()
            assert len(meta)==p.shape[0]
            rep=meta.loc[meta.index.repeat(NODES)].reset_index(drop=True)
            rep["node_index"]=np.tile(np.arange(NODES),p.shape[0])
            rep["split"]=split
            rep["model"]="GCN"
            rep["topology"]=topology
            rep["seed"]=seed
            rep["y_true"]=y.reshape(-1)
            rep["y_pred"]=p.reshape(-1)
            rep["error"]=rep["y_pred"]-rep["y_true"]
            all_preds.append(rep)

metrics_df=pd.DataFrame(all_metrics)
pred_df=pd.concat(all_preds,ignore_index=True)
metrics_df.to_csv(OUT/"02_metrics_all_topologies_seeds_splits.csv",index=False)
pred_df.to_csv(OUT/"03_predictions_all_topologies_seeds.csv",index=False)

summary=(metrics_df.groupby(["topology","split"])
         .agg(
             n_seeds=("seed","nunique"),
             MAE_mean=("MAE","mean"), MAE_SD=("MAE","std"),
             RMSE_mean=("RMSE","mean"), RMSE_SD=("RMSE","std"),
             R2_mean=("R2","mean"), R2_SD=("R2","std"),
             Bias_mean=("Bias","mean"), Bias_SD=("Bias","std")
         ).reset_index())
summary.to_csv(OUT/"04_metrics_mean_sd_across_seeds.csv",index=False)

# Paired topology comparison within each seed on TEST
testp=pred_df[pred_df["split"]=="test"].copy()
paired=[]
for seed in SEEDS:
    d=testp[testp.seed==seed].copy()
    # row order is guaranteed by construction within each topology
    wide=d.pivot_table(
        index=["node_index"] + [c for c in ["timestamp","target_timestamp"] if c in d.columns],
        columns="topology", values="error", aggfunc="first"
    ).dropna()
    for topo in ["G_operational","G_shuffled"]:
        a=np.abs(wide[topo].to_numpy())
        b=np.abs(wide["G_native"].to_numpy())
        paired.append({
            "seed":seed,
            "comparison":f"{topo}_vs_G_native",
            "n":len(a),
            "mean_abs_error_difference_C":float(np.mean(a-b)),
            "pct_first_topology_better":float(100*np.mean(a<b))
        })
    a=np.abs(wide["G_operational"].to_numpy())
    b=np.abs(wide["G_shuffled"].to_numpy())
    paired.append({
        "seed":seed,
        "comparison":"G_operational_vs_G_shuffled",
        "n":len(a),
        "mean_abs_error_difference_C":float(np.mean(a-b)),
        "pct_first_topology_better":float(100*np.mean(a<b))
    })
pd.DataFrame(paired).to_csv(OUT/"05_TEST_paired_topology_comparisons.csv",index=False)

# Degree/topology audit
audit=[]
for topo,e in edges.items():
    audit.append({
        "topology":topo,
        "stored_edge_entries":e.shape[1],
        "unique_undirected_interdevice_edges":len(undirected_pairs(e)),
        "degree_vector":";".join(map(str,degree_vector(e).tolist()))
    })
pd.DataFrame(audit).to_csv(OUT/"06_topology_audit.csv",index=False)

manifest={
    "protocol_status":"FROZEN_07D_INPUTS",
    "task":"indoor temperature t+15 min",
    "graph_tensor":"5x7x3",
    "node_level_counts":EXPECTED_NODE_LEVEL,
    "snapshot_counts":EXPECTED_SNAPSHOTS,
    "features":["temperature","relative_humidity","co2"],
    "topologies":{
        "G_native":"no inferred inter-device edges; self-loops added by GCN",
        "G_operational":"train-only operational similarity topology frozen in 07D",
        "G_shuffled":"degree-preserving shuffled control frozen in 07D"
    },
    "spatial_uuid_ifc_mapping_claim":False,
    "architecture_selection":"hidden width selected on G_operational validation RMSE only, seed 42",
    "selected_configuration":SELECTED_NAME,
    "final_training_seeds":SEEDS,
    "test_used_for_selection":False,
    "batch_size":BATCH_SIZE,
    "max_epochs":MAX_EPOCHS,
    "patience":PATIENCE,
    "optimizer":"Adam",
    "learning_rate":LR,
    "weight_decay":WEIGHT_DECAY,
    "python":platform.python_version(),
    "torch":torch.__version__,
    "device":str(device),
    "torch_geometric_required":False
}
with open(OUT/"07_reproducibility_manifest.json","w") as f:
    json.dump(manifest,f,indent=2)

print("\n[3/3] SUMMARY")
print(summary.to_string(index=False))
print("\nVALIDATION")
print("✓ Frozen 07D graph tensors used")
print("✓ Exact graph snapshots 9,243 / 1,975 / 1,970")
print("✓ Exact node predictions 64,701 / 13,825 / 13,790")
print("✓ Same 5×3 sensor history per node")
print("✓ G_native contains no inferred inter-device edges")
print("✓ G_operational topology remains train-only")
print("✓ G_shuffled preserves operational labeled node degrees")
print("✓ Same selected GCN architecture used for all three topologies")
print("✓ Architecture selected on validation only")
print("✓ TEST never used for architecture/topology selection")
print("✓ Three predefined final seeds: 42, 123, 2026")
print("✓ No UUID→IfcSpace spatial claim")
print("✓ Predictions saved for later paired/block-bootstrap analysis")
